### TorchAudio basics ###

In [ ]:
# import os

# os.environ['KMP_DUPLICATE_LIB_OK']='True'


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm # For a nice progress bar
from torchvision import datasets
import torchvision.transforms as transforms
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
import librosa
import requests



### Train a Speech Model ###

Now we will use the above-learned concepts to train a speech model. Specifically, we will create a speech command classification model. We will train the audio classification model on the SpeechCommands dataset. This dataset comprises audio samples of 35 commands spoken by different people.

We will first extract the audio files and their respective labels to prepare the dataset for training. We will then perform the necessary transformations to convert the audio files to spectrogram and MFCC features.

In [2]:
# load the audio files from the dataset folder using the load_audio_files which is defined below

def load_audio_files(data_path, labels):

    dataset = []
    walker = sorted(str(p) for p in Path(data_path).glob(f'*.wav'))

    for i, file_path in enumerate(walker):
        path, filename = os.path.split(file_path)
        speaker, _ = os.path.splitext(filename)
        speaker_id, utterance_number = speaker.split("_nohash_")
        utterance_number = int(utterance_number)
        waveform, sample_rate = torchaudio.load(file_path)
        dataset.append([waveform, sample_rate, labels, speaker_id, utterance_number])
        
    return dataset


yes_commands = load_audio_files('./speechcommands_small/yes', 'yes')
no_commands = load_audio_files('./speechcommands_small/no', 'no')

print(f'Length of yes dataset: {len(yes_commands)}')
print(f'Length of no dataset: {len(no_commands)}')


Length of yes dataset: 4044
Length of no dataset: 3941


In [5]:
# Create the dataloaders

trainloader_yes = torch.utils.data.DataLoader(yes_commands, batch_size=1, shuffle=True)
trainloader_no = torch.utils.data.DataLoader(no_commands, batch_size=1, shuffle=True)


In [6]:
# Transform our audio data to their respective spectrograms using the function audio_to_spec. 
# This function utilizes the torchaudio.transforms module to create the spectrogram. 
# We save the spectrogram as images using matplotlib.pyplot in a new directory ./data/spectrograms/{label_dir}/

def audio_to_spec(trainloader, label_dir):
    # Create the destination directory before the loop
    save_path = f'./data/spectrograms/{label_dir}'
    os.makedirs(save_path, exist_ok=True) 
           
    for i, data in enumerate(trainloader):
        waveform = data[0]
        sample_rate = data[1][0]
        label = data[2]
        ID = data[3]

        # create transformed waveforms
        spectrogram_tensor = torchaudio.transforms.Spectrogram()(waveform)     
        
        fig = plt.figure()
        plt.imsave(f'./data/spectrograms/{label_dir}/spec_img{i}.png', spectrogram_tensor[0].log2()[0,:,:].numpy(), cmap='viridis')
        plt.close()

audio_to_spec(trainloader_yes, 'train/yes')
audio_to_spec(trainloader_no, 'train/no')


/Users/jagannathrao/Library/CloudStorage/OneDrive-UniversityofGeorgia/UGA_Courses/INFO_NEW_COURSES_FROM_FALL_2022/INFO_4000/INFO_4000_Fall_2026/Weekly_Content/.venv/lib/python3.13/site-packages/matplotlib/colors.py:2559: RuntimeWarning: invalid value encountered in subtract
  resdat -= vmin
/Users/jagannathrao/Library/CloudStorage/OneDrive-UniversityofGeorgia/UGA_Courses/INFO_NEW_COURSES_FROM_FALL_2022/INFO_4000/INFO_4000_Fall_2026/Weekly_Content/.venv/lib/python3.13/site-packages/matplotlib/colors.py:2560: RuntimeWarning: invalid value encountered in divide
  resdat /= (vmax - vmin)


#### We shall use VGG to train the model on these spectrograms

In [23]:
# VGG16 expects 224x224 images and specific normalization values
# def repeat_channels(x: torch.Tensor) -> torch.Tensor:
#     """
#     Takes a 1-channel tensor and repeats it three times to create a 3-channel tensor.
#     """
#     return x.repeat(3, 1, 1)

# VGG16 expects 224x224 images and specific normalization values
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --- Update the transform pipelines to use the named function ---
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [24]:
# create the dataloader which is responsible for feeding data to the model for training, 
# this is done using the torch.utils.data.datasets.ImageFolder module. 
# This module loads the spectrogram images directly to a dataset object.

data_path = './data/spectrograms/train/'

# 1. Create two dataset instances from the same folder, each with a different transform
train_dataset_obj = datasets.ImageFolder(root=data_path, transform=train_transforms)
test_dataset_obj = datasets.ImageFolder(root=data_path, transform=test_transforms)

class_map = train_dataset_obj.class_to_idx

# 2. Get the indices and split them, instead of splitting the dataset object directly
dataset_size = len(train_dataset_obj)
indices = list(range(dataset_size))
split = int(np.floor(0.8 * dataset_size)) # 80% for training
np.random.seed(42) # for reproducibility
np.random.shuffle(indices)
train_indices, test_indices = indices[:split], indices[split:]

# 3. Create the Subset for train and test using the appropriate dataset object and indices
yes_no_train_dataset = torch.utils.data.Subset(train_dataset_obj, train_indices)
yes_no_test_dataset = torch.utils.data.Subset(test_dataset_obj, test_indices)

print("\nClass category and index of the images: {}\n".format(class_map))



Class category and index of the images: {'no': 0, 'yes': 1}



In [25]:
print("Training size:", len(yes_no_train_dataset))
print("Testing size:",len(yes_no_test_dataset))




Training size: 6388
Testing size: 1597


In [26]:
# --- 1. Model Definition ---
def build_model(num_classes=2):
    """
    Builds a VGG16 model with a modified classifier for 2 classes.
    The convolutional layers are frozen.
    """
    # Load a pre-trained VGG16 model
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

    # Freeze all the parameters in the feature extraction part (the CNN layers)
    for param in model.features.parameters():
        param.requires_grad = False
    
    print("✅ Convolutional layers frozen.")

    # Get the number of input features for the classifier
    num_ftrs = model.classifier[6].in_features

    # Replace the last layer with a new one for our 2 classes
    # The new layer's parameters will have requires_grad=True by default
    model.classifier[6] = nn.Linear(num_ftrs, num_classes)
    
    print(f"✅ Final classifier layer replaced for {num_classes} classes.")
    
    return model

# --- 2. Training and Testing Loops ---

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    Runs a single training epoch.
    """
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Use tqdm for a progress bar
    for inputs, labels in tqdm(train_loader, desc="Training"):
        # Move data to the selected device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100
    return epoch_loss, epoch_acc


def test_loop(model, test_loader, criterion, device):
    """
    Runs evaluation on the test set.
    """
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # No need to track gradients for testing
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100
    return epoch_loss, epoch_acc




In [27]:
# Automatically pick the best available hardware accelerator
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🚀 Using device: {device.type.upper()}")
#device ='cpu'  # Force CPU for this example, remove this line to use GPU if available

🚀 Using device: MPS


In [28]:
# --- Main Execution Block ---

# --- 3. Hyperparameters and Setup ---
NUM_EPOCHS = 5
LEARNING_RATE = 0.001
BATCH_SIZE = 32 # The batch size you used for your DataLoaders
NUM_CLASSES = 2 # "yes" and "no"


# --- 4. Prepare DataLoaders (Replace with your actual DataLoaders) ---
train_dataloader = torch.utils.data.DataLoader(
    yes_no_train_dataset,
    batch_size=16,
    num_workers=0,
    shuffle=True,
)

test_dataloader = torch.utils.data.DataLoader(
    yes_no_test_dataset,
    batch_size=16,
    num_workers=0,
    shuffle=True,
)

# --- 5. Initialize Model, Loss, and Optimizer ---
model = build_model(num_classes=NUM_CLASSES).to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 6. Start Training and Testing ---
for epoch in range(NUM_EPOCHS):
    print(f"--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    
    train_loss, train_acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    
    test_loss, test_acc = test_loop(model, test_dataloader, criterion, device)
    print(f"Epoch {epoch+1} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
    print("-" * 50)

print("🏁 Training Finished!")

✅ Convolutional layers frozen.
✅ Final classifier layer replaced for 2 classes.
--- Epoch 1/5 ---


Training: 100%|██████████| 400/400 [03:27<00:00,  1.93it/s]


Epoch 1 | Train Loss: 0.7437 | Train Acc: 75.03%


Testing: 100%|██████████| 100/100 [00:33<00:00,  3.03it/s]


Epoch 1 | Test Loss: 0.2499 | Test Acc: 88.85%
--------------------------------------------------
--- Epoch 2/5 ---


Training:   8%|▊         | 33/400 [00:19<03:37,  1.69it/s]


KeyboardInterrupt: 